# Análise de Atenção Temporal — ReferenceEncoder VAE-GST

Este notebook extrai e interpreta os **pesos de atenção temporal** do `ReferenceEncoder` modificado.

O encoder agora usa *attention pooling* em vez de pegar apenas o último hidden state da GRU:
```
GRU(t) → score(t) → softmax → soma ponderada
```
Cada peso `w(t)` indica **quanto aquele frame de mel da referência influenciou o embedding de estilo**.

**Perguntas que vamos responder:**
1. O modelo foca em quais partes do áudio de referência?
2. A atenção correlaciona com energia, pitch ou estrutura espectral?
3. Referências diferentes produzem padrões de atenção diferentes?
4. O que os embeddings de estilo aprenderam?

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'training' / 'training-tacotron2-vae'))
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'data' / 'loader_vae_tacotron'))

import json
import numpy as np
import torch
import torchaudio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm

from models.tacotron2_vae.hparams import create_hparams
from models.tacotron2_vae.model import load_tacotron2_vae_model
from models.tacotron2_vae.layers import TacotronSTFT
from text_processing import TextProcessor
from train_utils import load_checkpoint, find_latest_checkpoint

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10
print('Imports OK')

## 1. Carregamento do Modelo

In [ ]:
EXPERIMENT = PROJECT_ROOT / 'experiments' / 'tacotron2-vae' / 'lj_speech_v1'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Carrega hparams e symbols
with open(EXPERIMENT / 'hparams.json') as f:
    hparams_dict = json.load(f)
hparams = create_hparams(hparams_dict)

with open(EXPERIMENT / 'symbols.json') as f:
    symbols_data = json.load(f)
text_processor = TextProcessor(
    symbols=symbols_data['symbols'],
    cleaner_names=symbols_data.get('cleaner_names', ['english_cleaners'])
)

# Carrega o modelo
model = load_tacotron2_vae_model(hparams, device=device)
ckpt = find_latest_checkpoint(EXPERIMENT / 'checkpoints')
model, _, _, iteration = load_checkpoint(ckpt, model)
model.eval()
print(f'Modelo carregado: {ckpt.name}  (iteration {iteration})')

## 2. Utilitários

In [ ]:
stft = TacotronSTFT(
    filter_length=1024, hop_length=256, win_length=1024,
    sampling_rate=22050, mel_fmin=0.0, mel_fmax=8000.0
).to(device)

def load_mel(wav_path: str):
    """Carrega .wav e retorna mel (1, n_mels, T) no device."""
    wav, sr = torchaudio.load(wav_path)
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    if sr != 22050:
        wav = torchaudio.functional.resample(wav, sr, 22050)
    wav = torch.clamp(wav / wav.abs().max(), -1.0, 1.0)
    return stft.mel_spectrogram(wav.to(device))  # (1, 80, T)

def get_attention(wav_path: str):
    """Retorna (mel_np, weights_np, z_np, mu_np) para um áudio de referência."""
    mel = load_mel(wav_path)
    with torch.no_grad():
        _, mu, _, z, weights = model.vae_gst.forward_with_attention(mel)
    return (
        mel.squeeze(0).cpu().numpy(),     # (80, T)
        weights.squeeze(0).cpu().numpy(), # (T',)  T' = T após convs
        z.squeeze(0).cpu().numpy(),       # (L,)
        mu.squeeze(0).cpu().numpy(),      # (L,)
    )

print('Utilitários prontos')

## 3. Referências e extração dos pesos

In [ ]:
RAW = PROJECT_ROOT / 'data' / 'raw' / 'LJSpeech-1.1' / 'wavs'

references = {
    'LJ001-0001': str(RAW / 'LJ001-0001.wav'),  # livro 1
    'LJ025-0010': str(RAW / 'LJ025-0010.wav'),  # livro 25
    'LJ050-0050': str(RAW / 'LJ050-0050.wav'),  # livro 50
    'LJ010-0001': str(RAW / 'LJ010-0001.wav'),  # livro 10
    'LJ035-0001': str(RAW / 'LJ035-0001.wav'),  # livro 35
}

# Textos dos áudios de referência
import csv
meta = {}
with open(PROJECT_ROOT / 'data' / 'raw' / 'LJSpeech-1.1' / 'metadata.csv') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) >= 3:
            meta[parts[0]] = parts[2]

data = {}
for name, path in references.items():
    mel, weights, z, mu = get_attention(path)
    data[name] = {'mel': mel, 'weights': weights, 'z': z, 'mu': mu, 'path': path}
    sr_ratio = mel.shape[1] / weights.shape[0]  # frames mel / frames GRU
    print(f'{name}: mel={mel.shape}  weights={weights.shape}  '
          f'(1 weight ≈ {sr_ratio:.1f} mel frames)  '
          f'texto: "{meta.get(name, "?")[:60]}"')

## 4. Pesos de atenção temporal

Cada barra mostra o **peso de atenção** que o modelo deu a cada frame da GRU ao codificar aquele áudio de referência.
Um pico alto significa: *"este trecho do áudio foi mais influente para o embedding de estilo."*

In [ ]:
fig, axes = plt.subplots(len(data), 1, figsize=(14, 2.2 * len(data)), sharex=False)

colors = plt.cm.tab10(np.linspace(0, 0.6, len(data)))

for ax, (name, d), color in zip(axes, data.items(), colors):
    w = d['weights']
    t = np.arange(len(w))
    ax.bar(t, w, color=color, alpha=0.85, width=0.9)
    ax.axhline(1/len(w), color='gray', linestyle='--', linewidth=0.8, label='atenção uniforme')
    peak_idx = np.argmax(w)
    ax.axvline(peak_idx, color='red', linestyle=':', linewidth=1.2)
    ax.set_ylabel('peso', fontsize=8)
    ax.set_title(f'{name} — pico em t={peak_idx} ({w[peak_idx]:.4f})', fontsize=9)
    ax.set_xlim(-0.5, len(w) - 0.5)
    # entropia dos pesos (quão concentrada é a atenção)
    entropy = -np.sum(w * np.log(w + 1e-9))
    ax.text(0.98, 0.88, f'entropia={entropy:.2f}', transform=ax.transAxes,
            ha='right', fontsize=8, color='dimgray')

axes[-1].set_xlabel('frame da GRU (após convoluções)')
fig.suptitle('Pesos de Atenção Temporal — ReferenceEncoder', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. Atenção sobreposta ao mel da referência

Aqui vemos **onde no espectrograma** o modelo prestou atenção.
A barra vermelha na parte de cima marca o frame GRU com maior peso.

In [ ]:
for name, d in data.items():
    mel = d['mel']       # (80, T_mel)
    w   = d['weights']   # (T_gru,)

    # Interpola os pesos GRU de volta para a escala temporal do mel
    T_mel = mel.shape[1]
    T_gru = len(w)
    w_interp = np.interp(
        np.linspace(0, T_gru - 1, T_mel),
        np.arange(T_gru), w
    )

    fig, (ax_mel, ax_w) = plt.subplots(
        2, 1, figsize=(13, 4),
        gridspec_kw={'height_ratios': [4, 1]}, sharex=True
    )

    # Mel spectrogram
    im = ax_mel.imshow(mel, origin='lower', aspect='auto',
                       interpolation='nearest', cmap='magma')
    # Overlay: highlight das regiões de alta atenção
    attn_norm = (w_interp - w_interp.min()) / (w_interp.max() - w_interp.min() + 1e-9)
    for t in range(T_mel):
        if attn_norm[t] > 0.6:  # top 40% de atenção
            ax_mel.axvline(t, color='cyan', alpha=attn_norm[t] * 0.4, linewidth=1)

    ax_mel.set_ylabel('Mel bin')
    ax_mel.set_title(f'{name}  |  "{meta.get(name, "")[:70]}"', fontsize=9)
    plt.colorbar(im, ax=ax_mel, pad=0.01)

    # Curva de atenção interpolada
    ax_w.fill_between(range(T_mel), w_interp, alpha=0.7, color='cyan')
    ax_w.axhline(1 / T_gru, color='gray', linestyle='--', linewidth=0.8)
    ax_w.set_ylabel('atenção', fontsize=8)
    ax_w.set_xlabel('frame mel')
    ax_w.set_xlim(0, T_mel - 1)

    plt.tight_layout()
    plt.show()

## 6. Correlação: atenção × energia do áudio

A hipótese é que o modelo pode estar usando regiões de **alta energia** (fala forte/clara) como âncoras de estilo. Vamos verificar calculando o RMS por frame e comparando com o perfil de atenção.

In [ ]:
fig, axes = plt.subplots(len(data), 1, figsize=(13, 2.5 * len(data)), sharex=False)

correlations = {}

for ax, (name, d) in zip(axes, data.items()):
    mel = d['mel']     # (80, T_mel) — log-mel
    w   = d['weights'] # (T_gru,)
    T_mel = mel.shape[1]

    # Energia: média das bandas mel por frame (proxy de energia log)
    energy = mel.mean(axis=0)  # (T_mel,)
    energy_norm = (energy - energy.min()) / (energy.max() - energy.min() + 1e-9)

    # Interpola atenção para escala mel
    T_gru = len(w)
    w_interp = np.interp(np.linspace(0, T_gru - 1, T_mel), np.arange(T_gru), w)
    w_norm = (w_interp - w_interp.min()) / (w_interp.max() - w_interp.min() + 1e-9)

    corr = np.corrcoef(w_norm, energy_norm)[0, 1]
    correlations[name] = corr

    t = np.arange(T_mel)
    ax.fill_between(t, energy_norm, alpha=0.4, color='orange', label='energia (norm)')
    ax.fill_between(t, w_norm,      alpha=0.6, color='steelblue', label='atenção (norm)')
    ax.set_title(f'{name}  —  correlação atenção×energia: r={corr:.3f}', fontsize=9)
    ax.set_ylabel('valor norm.')
    ax.set_xlim(0, T_mel - 1)
    if ax == axes[0]:
        ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('frame mel')
fig.suptitle('Atenção temporal × Energia do áudio', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('\nCorrelações atenção × energia:')
for name, r in correlations.items():
    print(f'  {name}: r={r:.3f}')

## 7. Correlação: atenção × centróide espectral

O **centróide espectral** indica o centro de massa em frequência — alto centróide = energia nas altas frequências (consoantes, sibilantes). Baixo centróide = vogais e som fundamental.

In [ ]:
# Frequências centrais de cada banda mel (aproximação linear)
n_mels = 80
mel_freqs = np.linspace(0, 8000, n_mels)

fig, axes = plt.subplots(len(data), 1, figsize=(13, 2.5 * len(data)), sharex=False)
centroid_corrs = {}

for ax, (name, d) in zip(axes, data.items()):
    mel = d['mel']      # (80, T)
    w   = d['weights']
    T_mel = mel.shape[1]

    # Centróide: soma ponderada por freq, normalizada pela energia total
    mel_linear = np.exp(mel)  # desfaz log
    centroid = (mel_linear * mel_freqs[:, None]).sum(0) / (mel_linear.sum(0) + 1e-9)
    centroid_norm = (centroid - centroid.min()) / (centroid.max() - centroid.min() + 1e-9)

    T_gru = len(w)
    w_interp = np.interp(np.linspace(0, T_gru - 1, T_mel), np.arange(T_gru), w)
    w_norm = (w_interp - w_interp.min()) / (w_interp.max() - w_interp.min() + 1e-9)

    corr = np.corrcoef(w_norm, centroid_norm)[0, 1]
    centroid_corrs[name] = corr

    t = np.arange(T_mel)
    ax.fill_between(t, centroid_norm, alpha=0.4, color='green', label='centróide espectral')
    ax.fill_between(t, w_norm,        alpha=0.6, color='steelblue', label='atenção')
    ax.set_title(f'{name}  —  r(atenção, centróide)={corr:.3f}', fontsize=9)
    ax.set_xlim(0, T_mel - 1)
    if ax == axes[0]:
        ax.legend(fontsize=8)

axes[-1].set_xlabel('frame mel')
fig.suptitle('Atenção × Centróide Espectral', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('\nCorrelações atenção × centróide espectral:')
for n, r in centroid_corrs.items():
    print(f'  {n}: r={r:.3f}')

## 8. Comparação dos embeddings de estilo (μ)

Visualizamos o vetor `μ` (média do VAE) para cada referência. Dimensões com grande variação entre referências são as mais **discriminativas de estilo**.

In [ ]:
names = list(data.keys())
mus = np.stack([data[n]['mu'] for n in names])  # (N_refs, L)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Heatmap dos μ ---
ax = axes[0]
im = ax.imshow(mus, aspect='auto', cmap='RdBu_r',
               norm=Normalize(vmin=-3, vmax=3))
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('dimensão latente (μ)')
ax.set_title('Vetores μ por referência')
plt.colorbar(im, ax=ax)

# --- Variância por dimensão (quais dims discriminam mais) ---
ax = axes[1]
var_per_dim = mus.var(axis=0)
top_k = np.argsort(var_per_dim)[::-1][:10]
ax.bar(range(len(top_k)), var_per_dim[top_k], color='steelblue')
ax.set_xticks(range(len(top_k)))
ax.set_xticklabels([f'dim {i}' for i in top_k], rotation=45, fontsize=8)
ax.set_ylabel('variância entre referências')
ax.set_title('Top-10 dimensões mais discriminativas')

plt.tight_layout()
plt.show()

print('Variância total entre referências:', var_per_dim.sum().round(4))
print(f'Top-5 dims discriminativas: {top_k[:5].tolist()}')

## 9. Ablação temporal: o que acontece com o embedding ao mascarar regiões?

Para entender **quais partes do áudio são causalmente importantes** para o embedding de estilo, mascaramos progressivamente os frames da GRU com peso zero e medimos o quanto o embedding muda (distância L2 vs. embedding original).

In [ ]:
def embed_with_mask(mel_tensor, mask):
    """Retorna embedding z com os frames GRU mascarados pelo mask booleano."""
    # Passa pelo CNN do reference encoder
    ref_enc = model.vae_gst.ref_encoder
    with torch.no_grad():
        gru_input = ref_enc._cnn_forward(mel_tensor)  # (1, T_gru, input_size)
        # Aplica máscara: zera frames selecionados
        masked = gru_input.clone()
        masked[:, mask, :] = 0.0
        enc_out, _ = ref_enc._encode(masked)
        _, _, _, z = model.vae_gst._vae_from_enc(enc_out)
    return z.squeeze(0).cpu().numpy()

fig, axes = plt.subplots(1, len(data), figsize=(4 * len(data), 4), sharey=True)

for ax, (name, d) in zip(axes, data.items()):
    mel_t = load_mel(d['path'])
    ref_enc = model.vae_gst.ref_encoder
    with torch.no_grad():
        gru_input = ref_enc._cnn_forward(mel_t)
    T_gru = gru_input.shape[1]

    # Embedding original
    z_orig = embed_with_mask(mel_t, np.zeros(T_gru, dtype=bool))

    # Mascara 1 frame por vez e mede delta
    deltas = []
    for t in range(T_gru):
        mask = np.zeros(T_gru, dtype=bool)
        mask[t] = True
        z_masked = embed_with_mask(mel_t, mask)
        deltas.append(np.linalg.norm(z_orig - z_masked))

    deltas = np.array(deltas)
    w = d['weights']

    ax.bar(range(T_gru), deltas / deltas.max(), alpha=0.7, color='coral', label='Δz (ablação)')
    ax.plot(range(T_gru), w / w.max(), color='steelblue', linewidth=1.5, label='peso atenção')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('frame GRU')
    if ax == axes[0]:
        ax.set_ylabel('valor norm.')
        ax.legend(fontsize=8)

fig.suptitle('Ablação temporal: atenção vs. impacto real no embedding', fontsize=11)
plt.tight_layout()
plt.show()

## 10. Resumo e Interpretação

Análise dos resultados obtidos nas seções anteriores.

In [ ]:
print('=' * 60)
print('RESUMO DA ANÁLISE DE ATENÇÃO TEMPORAL')
print('=' * 60)

for name, d in data.items():
    w = d['weights']
    entropy = -np.sum(w * np.log(w + 1e-9))
    max_entropy = np.log(len(w))
    concentration = 1 - (entropy / max_entropy)  # 0=difusa, 1=concentrada
    peak_pos_pct  = np.argmax(w) / len(w) * 100

    print(f'\n{name}')
    print(f'  Frames GRU      : {len(w)}')
    print(f'  Entropia        : {entropy:.3f}  (max={max_entropy:.3f})')
    print(f'  Concentração    : {concentration:.2%}  (1=totalmente focado num frame)')
    print(f'  Pico de atenção : {peak_pos_pct:.0f}% do áudio (frame {np.argmax(w)})')
    print(f'  r(attn,energia) : {correlations.get(name, float("nan")):.3f}')
    print(f'  r(attn,centróide): {centroid_corrs.get(name, float("nan")):.3f}')

print('\n' + '=' * 60)
print('INTERPRETAÇÃO GERAL')
print('=' * 60)
mean_energy_corr   = np.mean(list(correlations.values()))
mean_centroid_corr = np.mean(list(centroid_corrs.values()))
print(f'Correlação média atenção×energia   : {mean_energy_corr:.3f}')
print(f'Correlação média atenção×centróide : {mean_centroid_corr:.3f}')

print()
if abs(mean_energy_corr) > abs(mean_centroid_corr):
    print('→ O modelo parece guiar a atenção mais por ENERGIA do que por centróide espectral.')
    print('  Isso faz sentido: regiões de fala clara/forte carregam mais informação prosódica.')
else:
    print('→ O modelo parece guiar a atenção mais por ESTRUTURA ESPECTRAL (centróide) do que energia.')
    print('  Isso sugere que consoantes/vogais específicas são mais informativas para o estilo.')

entropies = [-np.sum(d['weights'] * np.log(d['weights'] + 1e-9)) for d in data.values()]
mean_ent = np.mean(entropies)
max_possible = np.log(list(data.values())[0]['weights'].shape[0])
print(f'\nEntropia média: {mean_ent:.2f} / {max_possible:.2f} ({mean_ent/max_possible:.0%} de difusão)')
if mean_ent / max_possible > 0.7:
    print('→ Atenção DIFUSA: o modelo distribuiu o peso por muitos frames.')
    print('  Isso indica que o estilo é capturado de forma global — não há um único instante dominante.')
else:
    print('→ Atenção CONCENTRADA: o modelo focou em poucos frames específicos.')
    print('  Isso indica que o estilo prosódico está concentrado em regiões específicas do áudio.')